<a href="https://colab.research.google.com/github/rajperfattah/Abdul_Portfolio/blob/main/Another_copy_of_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

!pip install transformers datasets rouge-score nltk scikit-learn

In [ ]:
import nltk
nltk.download('punkt')

# Sample text for summarization testing
sample_text = """
Natural Language Processing (NLP) is a subfield of artificial intelligence, computer science,
and linguistics focused on the interactions between computers and human language.
It is increasingly critical in modern applications as the volume of digital text data grows exponentially.
Automatic text summarization aims to address this information overload by generating concise and
meaningful summaries of large documents while preserving their core semantics and essential information.
There are generally two main approaches to text summarization: extractive and abstractive.
Extractive methods select important sentences directly from the original text based on statistical
or graph-based rankings like TF-IDF or TextRank. On the other hand, abstractive methods use advanced
deep learning transformer architectures like BART or T5 to generate new paraphrased sentences,
resembling human-like summaries. Evaluating these summaries is typically performed using automated
metrics such as ROUGE scores, which measure the overlap between generated summaries and reference summaries.
"""
print("Sample text loaded successfully! Length of text:", len(sample_text))

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Sample text loaded successfully! Length of text: 1090


True

In [ ]:
import nltk
from transformers import pipeline
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- 1. Extractive Approach (TF-IDF based sentence ranking) ---")
sentences = nltk.sent_tokenize(sample_text)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(sentences)
sentence_scores = np.array(X.sum(axis=1)).ravel()
ranked_sentence_indices = sentence_scores.argsort()[::-1]
# Select top 2 sentences for extractive summary
top_sentences = sorted(ranked_sentence_indices[:2])
extractive_summary = " ".join([sentences[i] for i in top_sentences])
print("Extractive Summary:\n", extractive_summary)

print("\n--- 2. Abstractive Approach (Using text-generation pipeline with BART) ---")
# Using the updated 'text-generation' pipeline for facebook/bart-large-cnn
summarizer = pipeline("text-generation", model="facebook/bart-large-cnn")
prompt = f"Summarize the following text:\n{sample_text}\nSummary:"
outputs = summarizer(prompt, max_new_tokens=60, do_sample=False)
abstractive_summary = outputs[0]['generated_text'].replace(prompt, "").strip()
print("Abstractive Summary:\n", abstractive_summary)

--- 1. Extractive Approach (TF-IDF based sentence ranking) ---
Extractive Summary:
 Automatic text summarization aims to address this information overload by generating concise and 
meaningful summaries of large documents while preserving their core semantics and essential information. On the other hand, abstractive methods use advanced 
deep learning transformer architectures like BART or T5 to generate new paraphrased sentences, 
resembling human-like summaries.

--- 2. Abstractive Approach (Using text-generation pipeline with BART) ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
mod

... (2 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Abstractive Summary:
 made made both had both both both made both both had had had both made made made had both had made made one both both one had both one both had one had had made both made with both both with both made had had one both one one made made with had both with with both one made


In [ ]:
from rouge_score import rouge_scorer

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Reference (ideal summary) vs Generated Summaries
reference_summary = "Automatic text summarization reduces information overload by extracting or generating concise summaries using extractive methods like TF-IDF or abstractive transformer models like BART, evaluated via ROUGE metrics."

print("--- Evaluating Extractive Summary ---")
ext_scores = scorer.score(reference_summary, extractive_summary)
for key, value in ext_scores.items():
    print(f"{key}: Precision={value.precision:.4f}, Recall={value.recall:.4f}, F1={value.fmeasure:.4f}")

print("\n--- Evaluating Abstractive Summary ---")
abs_scores = scorer.score(reference_summary, abstractive_summary)
for key, value in abs_scores.items():
    print(f"{key}: Precision={value.precision:.4f}, Recall={value.recall:.4f}, F1={value.fmeasure:.4f}")

--- Evaluating Extractive Summary ---
rouge1: Precision=0.3333, Recall=0.6071, F1=0.4304
rouge2: Precision=0.1200, Recall=0.2222, F1=0.1558
rougeL: Precision=0.2549, Recall=0.4643, F1=0.3291

--- Evaluating Abstractive Summary ---
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rouge2: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000
